# RelCheck v7 — Viability Test (15 images)
## Multi-Prompt BLIP-2 + SAM Masks + Contrastive Prompting + Visual Evidence Prompting

**New in v7 (over v6):**
1. **SAM masks** — facebook/sam-vit-base segments subject+object pixels; mask overlap confirms physical contact (holding, touching, eating)
2. **Contrastive prompting** — for each triple, Maverick answers BOTH the claim AND a foil relation (e.g., "riding" vs "standing next to"). Divergence = confidence.
3. **Visual evidence prompting** — Maverick describes *what it sees* before giving verdict. Observation text flows into Llama corrector as rich evidence.

**Architecture (5 stages):**
- BLIP-2 (multi-prompt) → Llama merge → caption
- Llama → triples + foil_relations
- GroundingDINO → bboxes → SAM → masks → geometry for SPATIAL
- Maverick → visual evidence + contrastive verdict for ACTION/ATTRIBUTE/fallback
- Llama text-only → targeted correction using Maverick's observation text


In [ ]:
# ============================================================
# Cell 1 — Install + Setup
# ============================================================
!pip install -q together transformers pillow torch torchvision
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

import os, json, base64, re, time, textwrap
from io import BytesIO
from collections import defaultdict
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from together import Together
from transformers import (
    Blip2Processor, Blip2ForConditionalGeneration,
    AutoProcessor, AutoModelForZeroShotObjectDetection,
    SamModel, SamProcessor,
)

# ── API Key ──
TOGETHER_API_KEY = ""  # <-- paste your key here
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY
client = Together(api_key=TOGETHER_API_KEY)

# ── Model IDs ──
VLM_MODEL  = "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"  # vision: verify
LLM_MODEL  = "meta-llama/Llama-3.3-70B-Instruct-Turbo"             # text: extract + correct
GDINO_ID   = "IDEA-Research/grounding-dino-tiny"
SAM_ID     = "facebook/sam-vit-base"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"VLM:    {VLM_MODEL}")
print(f"LLM:    {LLM_MODEL}")
print(f"GDino:  {GDINO_ID}")
print(f"SAM:    {SAM_ID}")


In [ ]:
# ============================================================
# Cell 2 — Load Models (BLIP-2 + GroundingDINO + SAM on GPU)
# ============================================================
# Runtime: ~4 min first run (downloading weights)

print("Loading BLIP-2...")
blip2_processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl", torch_dtype=torch.float16
).to(DEVICE)
print("  BLIP-2 loaded.")

print("Loading GroundingDINO...")
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(DEVICE)
print("  GroundingDINO loaded.")

print("Loading SAM (vit-base)...")
sam_model = SamModel.from_pretrained(SAM_ID).to(DEVICE)
sam_processor = SamProcessor.from_pretrained(SAM_ID)
print("  SAM loaded.")

print(f"\nAll models on {DEVICE}. Ready.")


In [ ]:
# ============================================================
# Cell 3 — Load 15 Test Images
# ============================================================
N_IMAGES = 15
CACHE_DIR = "/content/drive/MyDrive/RelCheck/nocaps_images"

import glob, random
all_paths = sorted(glob.glob(f"{CACHE_DIR}/*.jpg") + glob.glob(f"{CACHE_DIR}/*.png"))
if not all_paths:
    raise FileNotFoundError(f"No images found in {CACHE_DIR}")

random.seed(42)
selected = random.sample(all_paths, min(N_IMAGES, len(all_paths)))

images = {}
for p in selected:
    img_id = os.path.splitext(os.path.basename(p))[0]
    images[img_id] = Image.open(p).convert("RGB")

print(f"Loaded {len(images)} images from {CACHE_DIR}")
for img_id in images:
    print(f"  {img_id}: {images[img_id].size}")


In [ ]:
# ============================================================
# Cell 4 — Stage 1: Multi-Prompt BLIP-2 Caption Generation
# ============================================================
CAPTION_PROMPTS = [
    "Describe this image in detail.",
    "Where are the objects located relative to each other?",
    "What are the people or animals doing?",
    "Describe the colors, sizes, and attributes of objects.",
]

MERGE_PROMPT = """Merge these image descriptions into ONE coherent paragraph.
Use ONLY information present in the descriptions — do not add anything new.
Remove repetition. Write as a single flowing paragraph.

Descriptions:
{descriptions}

Merged paragraph:"""


def blip2_caption(image, prompt):
    inputs = blip2_processor(image, text=prompt, return_tensors="pt").to(DEVICE, torch.float16)
    with torch.no_grad():
        out = blip2_model.generate(**inputs, max_new_tokens=80)
    return blip2_processor.decode(out[0], skip_special_tokens=True).strip()


captions = {}
for img_id, img in images.items():
    sub_captions = [blip2_caption(img, p) for p in CAPTION_PROMPTS]

    numbered = "\n".join(f"{i+1}. {c}" for i, c in enumerate(sub_captions))
    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": MERGE_PROMPT.format(descriptions=numbered)}],
            temperature=0.0, max_tokens=200,
        )
        caption = resp.choices[0].message.content.strip()
        caption = re.sub(r'^(Merged paragraph|Revised paragraph|Here is[^:]*)[:\s]+', '',
                         caption, flags=re.IGNORECASE).strip()
        if caption.startswith('"'): caption = caption.strip('"')
    except Exception as e:
        caption = sub_captions[0]
        print(f"  Merge failed for {img_id}: {e}")

    captions[img_id] = caption
    print(f"[{img_id}]")
    for i, sc in enumerate(sub_captions):
        print(f"  P{i+1}: {sc}")
    print(f"  Merged: {caption[:150]}...")
    print()

print(f"Generated {len(captions)} captions.")


In [ ]:
# ============================================================
# Cell 5 — Stage 2: Triple Extraction + Foil Relation Generation
# ============================================================
# Foil = a plausible alternative relation (for contrastive verification).
# E.g., if relation="riding", foil_relation="standing next to"
# Both are sent to Maverick in Cell 7; divergence = high confidence.

TRIPLE_EXTRACTION_PROMPT = """Extract ALL relational triples from this caption as JSON array.

Each triple must have:
- subject: the entity doing or having the relation
- relation: the verb/preposition describing the relation
- object: the entity being related to
- type: one of SPATIAL | ACTION | ATTRIBUTE
- foil_relation: a plausible ALTERNATIVE relation that could be confused with this one
  (e.g., if relation="riding", foil="standing next to"; if relation="on", foil="beside")

Type rules:
- SPATIAL: physical position (on, above, below, next to, behind, near, under, inside, etc.)
- ACTION: interaction/activity (holding, riding, eating, wearing, playing with, using, etc.)
- ATTRIBUTE: property/quality (is red, is large, is wooden, looks old, etc.)

IMPORTANT:
- Every triple MUST have non-null subject, relation, AND object
- For ATTRIBUTE: subject=entity, relation=attribute description, object=value (e.g., "color", "red")
- Keep subject/object as simple noun phrases (2-4 words max)
- foil_relation must be a short verb/preposition phrase (not a full sentence)

Output ONLY valid JSON array. No explanation.

Caption: \"{caption}\"

JSON:"""


def extract_triples(caption):
    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": TRIPLE_EXTRACTION_PROMPT.format(caption=caption)}],
            temperature=0.0, max_tokens=1500,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r'^```json\s*', '', raw); raw = re.sub(r'```\s*$', '', raw)
        # Truncation recovery
        last_brace = raw.rfind("}")
        if last_brace != -1 and not raw.rstrip().endswith("]"):
            raw = raw[:last_brace + 1] + "]"
        triples = json.loads(raw)
        # Validate
        valid = []
        for t in triples:
            if all(t.get(k) for k in ("subject", "relation", "object", "type")):
                t.setdefault("foil_relation", "not present")
                valid.append(t)
        return valid
    except Exception as e:
        print(f"  Triple extraction failed: {e}")
        return []


all_triples = {}
for img_id, caption in captions.items():
    triples = extract_triples(caption)
    all_triples[img_id] = triples
    print(f"[{img_id}] {len(triples)} triples:")
    for t in triples:
        print(f"  ({t['subject']}, {t['relation']}, {t['object']}) [{t['type']}] foil='{t.get('foil_relation','?')}'")
    print()

total = sum(len(v) for v in all_triples.values())
print(f"Total triples: {total} across {len(all_triples)} images ({total/max(len(all_triples),1):.1f}/image)")


In [ ]:
# ============================================================
# Cell 6 — Stage 3a: GroundingDINO Detection + SAM Masks + Spatial Geometry
# ============================================================
# NEW in v7: SAM segments subject/object pixels.
# mask_overlap_ratio is stored per triple for use in Cell 7.

DETECTION_THRESHOLD = 0.25


def detect_objects(image, text_queries, threshold=DETECTION_THRESHOLD):
    text = ". ".join(text_queries) + "."
    inputs = gdino_processor(images=image, text=text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    results = gdino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=threshold, text_threshold=threshold,
        target_sizes=[image.size[::-1]],
    )[0]
    detections = []
    W, H = image.size
    lkey = "text_labels" if "text_labels" in results else "labels"
    for score, label, box in zip(results["scores"], results[lkey], results["boxes"]):
        x1, y1, x2, y2 = box.tolist()
        bbox_norm = [x1/W, y1/H, x2/W, y2/H]
        detections.append((label, score.item(), bbox_norm))
    return detections


def get_sam_mask(image, bbox_norm):
    """Get SAM pixel mask for a detected object given its normalized bbox."""
    W, H = image.size
    x1, y1, x2, y2 = bbox_norm
    box_px = [[int(x1*W), int(y1*H), int(x2*W), int(y2*H)]]
    try:
        inputs = sam_processor(image, input_boxes=[[box_px]], return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = sam_model(**inputs)
        masks = sam_processor.image_processor.post_process_masks(
            outputs.pred_masks.cpu(),
            inputs["original_sizes"].cpu(),
            inputs["reshaped_input_sizes"].cpu(),
        )
        scores = outputs.iou_scores[0, 0]
        best_idx = scores.argmax().item()
        return masks[0][0][best_idx].numpy().astype(bool)  # (H, W) bool mask
    except Exception as e:
        return None


def mask_overlap_ratio(mask1, mask2):
    """Fraction of smaller mask that overlaps with larger mask."""
    intersection = (mask1 & mask2).sum()
    smaller = min(mask1.sum(), mask2.sum())
    return float(intersection) / float(smaller) if smaller > 0 else 0.0


def find_best_detection(detections, query):
    query_lower = query.lower().strip()
    query_words = set(query_lower.split())
    best, best_score = None, -1
    for label, score, bbox in detections:
        ll = label.lower().strip()
        lw = set(ll.split())
        match = (query_lower == ll or query_words & lw or
                 (len(query_lower) > 4 and (query_lower in ll or ll in query_lower)))
        if match and score > best_score:
            best, best_score = (label, score, bbox), score
    return best


def check_spatial_relation(subj_box, obj_box, relation):
    sx1,sy1,sx2,sy2 = subj_box; ox1,oy1,ox2,oy2 = obj_box
    s_cx,s_cy = (sx1+sx2)/2,(sy1+sy2)/2
    o_cx,o_cy = (ox1+ox2)/2,(oy1+oy2)/2
    ix1,iy1 = max(sx1,ox1),max(sy1,oy1)
    ix2,iy2 = min(sx2,ox2),min(sy2,oy2)
    inter = max(0,ix2-ix1)*max(0,iy2-iy1)
    area_s = max((sx2-sx1)*(sy2-sy1),1e-8)
    area_o = max((ox2-ox1)*(oy2-oy1),1e-8)
    union = area_s+area_o-inter
    iou = inter/union if union>0 else 0
    rel = relation.lower().strip()
    if rel in ("on","on top of","atop"):
        h_overlap = max(0,min(sx2,ox2)-max(sx1,ox1))
        h_denom = min(sx2-sx1,ox2-ox1)
        h_ratio = h_overlap/h_denom if h_denom>1e-8 else 0
        if h_ratio < 0.15: return False
        subj_contained = (sx1>=ox1-0.05 and sx2<=ox2+0.05 and sy1>=oy1-0.05 and sy2<=oy2+0.05)
        return ((s_cy<o_cy and h_ratio>0.2) or abs(sy2-oy1)<0.15 or iou>0.1 or
                (subj_contained and s_cy<o_cy+0.1))
    elif rel in ("above","over"): return s_cy < o_cy
    elif rel in ("below","under","beneath","underneath"): return s_cy > o_cy
    elif rel in ("next to","beside","near"):
        return ((s_cx-o_cx)**2+(s_cy-o_cy)**2)**0.5 < 0.5 and iou < 0.3
    elif rel in ("in","inside"): return inter/area_s > 0.3 or iou > 0.15
    elif rel in ("left of","to the left of"): return s_cx < o_cx
    elif rel in ("right of","to the right of"): return s_cx > o_cx
    else: return None


# ── Run detection + SAM ──
spatial_results = {}
all_detections = {}
all_masks = {}

for img_id, triples in all_triples.items():
    img = images[img_id]
    W, H = img.size

    all_queries = set()
    for t in triples:
        if t.get("subject"): all_queries.add(t["subject"].lower())
        if t.get("object"): all_queries.add(t["object"].lower())

    detections = detect_objects(img, list(all_queries))
    all_detections[img_id] = detections

    # Get SAM masks for each detection
    masks = {}
    for label, score, bbox in detections:
        mask = get_sam_mask(img, bbox)
        if mask is not None:
            masks[label] = mask
    all_masks[img_id] = masks

    print(f"\n[{img_id}] {len(detections)} detections, {len(masks)} SAM masks")

    results = []
    for t in triples:
        if t["type"] != "SPATIAL":
            # Compute mask overlap for action triples (stored for Cell 7)
            subj_det = find_best_detection(detections, t["subject"])
            obj_det  = find_best_detection(detections, t["object"])
            m_overlap = None
            if subj_det and obj_det:
                sm = masks.get(subj_det[0])
                om = masks.get(obj_det[0])
                if sm is not None and om is not None:
                    m_overlap = mask_overlap_ratio(sm, om)
            results.append((t, "SKIP", f"mask_overlap={m_overlap:.3f}" if m_overlap is not None else "No detection"))
            continue

        subj_det = find_best_detection(detections, t["subject"])
        obj_det  = find_best_detection(detections, t["object"])

        if subj_det is None or obj_det is None:
            missing = [x for x, d in [(t["subject"],subj_det),(t["object"],obj_det)] if d is None]
            results.append((t, "FALLBACK_VLM", f"Detection failed: {missing}"))
            print(f"  ({t['subject']},{t['relation']},{t['object']}) -> FALLBACK (missing {missing})")
            continue

        geo = check_spatial_relation(subj_det[2], obj_det[2], t["relation"])
        if geo is None:
            results.append((t, "FALLBACK_VLM", f"No geo rule for '{t['relation']}'"))
            print(f"  ({t['subject']},{t['relation']},{t['object']}) -> FALLBACK (no rule)")
        elif geo:
            ev = f"Bbox geometry confirms '{t['relation']}'. Subj={[round(x,2) for x in subj_det[2]]} Obj={[round(x,2) for x in obj_det[2]]}"
            results.append((t, "SUPPORTED", ev))
            print(f"  ({t['subject']},{t['relation']},{t['object']}) -> SUPPORTED (geometry)")
        else:
            ev = f"Bbox geometry CONTRADICTS '{t['relation']}'. Subj={[round(x,2) for x in subj_det[2]]} Obj={[round(x,2) for x in obj_det[2]]}"
            results.append((t, "HALLUCINATED", ev))
            print(f"  ({t['subject']},{t['relation']},{t['object']}) -> HALLUCINATED (geometry)")

    spatial_results[img_id] = results

print("\n=== Detection + SAM Complete ===")


In [ ]:
# ============================================================
# Cell 7 — Stage 3b: Contrastive + Visual Evidence VLM Verification
# ============================================================
# THREE NEW TECHNIQUES over v6:
#
# 1. VISUAL EVIDENCE PROMPTING — Maverick describes what it sees
#    (OBSERVATION) before giving yes/no. Observation flows into corrector.
#
# 2. CONTRASTIVE PROMPTING — Also ask Maverick about the foil_relation.
#    Main=yes + Foil=no → SUPPORTED (high confidence)
#    Main=no + Foil=yes → HALLUCINATED (high confidence)
#    Both same → UNCERTAIN
#
# 3. SAM MASK OVERRIDE — For touching/holding/eating, if mask overlap
#    is significant, override to SUPPORTED (physical contact confirmed).

SAM_OVERLAP_THRESHOLD = 0.05   # >5% mask overlap = physical contact
SAM_CONTACT_RELATIONS = {"holding", "touching", "carrying", "eating",
                          "using", "gripping", "grasping", "wearing"}


def encode_image_base64(image):
    buf = BytesIO()
    image.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def format_detection_context(detections, subject, obj):
    subj_det = find_best_detection(detections, subject)
    obj_det  = find_best_detection(detections, obj)
    parts = []
    if subj_det:
        parts.append(f"'{subject}' detected at bbox {[round(x,2) for x in subj_det[2]]} (conf={subj_det[1]:.2f})")
    else:
        parts.append(f"'{subject}' NOT detected by GroundingDINO")
    if obj_det:
        parts.append(f"'{obj}' detected at bbox {[round(x,2) for x in obj_det[2]]} (conf={obj_det[1]:.2f})")
    else:
        parts.append(f"'{obj}' NOT detected by GroundingDINO")
    return "Detection evidence: " + "; ".join(parts) + "."


EVIDENCE_PROMPT = """Look at this image carefully.

{detection_context}

Claim to verify: "{subject} is {relation} {object}"

Step 1 — OBSERVATION: In 1-2 sentences, describe specifically what you see regarding how {subject} and {object} relate in this image.
Step 2 — VERDICT: Is the claim accurate? Answer ONLY yes or no.

Format your response exactly as:
OBSERVATION: [your description]
VERDICT: yes/no"""

FOIL_PROMPT = """Look at this image.

{detection_context}

Is this claim accurate about the image?
Claim: "{subject} is {foil_relation} {object}"

Answer ONLY yes or no."""


def call_vlm(image_b64, prompt, max_tokens=120):
    resp = client.chat.completions.create(
        model=VLM_MODEL,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}},
        ]}],
        temperature=0.0, max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()


def parse_evidence_response(raw):
    """Parse OBSERVATION + VERDICT from visual evidence response."""
    observation, verdict_raw = "", ""
    for line in raw.split("\n"):
        line = line.strip()
        if line.upper().startswith("OBSERVATION:"):
            observation = line[len("OBSERVATION:"):].strip()
        elif line.upper().startswith("VERDICT:"):
            verdict_raw = line[len("VERDICT:"):].strip().lower()
    verdict_yes = "yes" in verdict_raw and "no" not in verdict_raw
    return observation, verdict_yes


def contrastive_verdict(main_yes, foil_yes):
    """Combine main claim + foil claim into a verdict."""
    if main_yes and not foil_yes:
        return "SUPPORTED", "high"      # clearly confirmed
    elif not main_yes and foil_yes:
        return "HALLUCINATED", "high"   # foil is more accurate
    elif not main_yes and not foil_yes:
        return "UNCERTAIN", "low"       # neither confirmed
    else:
        return "UNCERTAIN", "low"       # contradiction (both yes)


# ── Run VLM verification ──
vlm_results = {}

for img_id in all_triples:
    img = images[img_id]
    b64 = encode_image_base64(img)
    detections = all_detections.get(img_id, [])
    masks = all_masks.get(img_id, {})
    results = []
    spatial_res = spatial_results.get(img_id, [])

    needs_vlm = [(t, ev) for t, verdict, ev in spatial_res
                 if verdict in ("SKIP", "FALLBACK_VLM")]

    print(f"\n[{img_id}] VLM verification: {len(needs_vlm)} triples")

    for t, fallback_reason in needs_vlm:
        subj, rel, obj = t["subject"], t["relation"], t["object"]
        foil = t.get("foil_relation", "not present")
        det_ctx = format_detection_context(detections, subj, obj)

        # ── 1. Visual Evidence Prompt ──
        ev_prompt = EVIDENCE_PROMPT.format(
            detection_context=det_ctx, subject=subj, relation=rel, object=obj)
        try:
            ev_raw = call_vlm(b64, ev_prompt, max_tokens=150)
            observation, main_yes = parse_evidence_response(ev_raw)
            if not observation:   # fallback if format not followed
                observation = ev_raw[:200]
                main_yes = "yes" in ev_raw.lower()
        except Exception as e:
            observation, main_yes = f"VLM error: {e}", False
        time.sleep(0.5)

        # ── 2. Contrastive Foil Prompt ──
        foil_yes = False
        if foil and foil != "not present":
            foil_prompt = FOIL_PROMPT.format(
                detection_context=det_ctx, subject=subj,
                foil_relation=foil, object=obj)
            try:
                foil_raw = call_vlm(b64, foil_prompt, max_tokens=10)
                foil_yes = "yes" in foil_raw.lower() and "no" not in foil_raw.lower()
            except Exception as e:
                foil_yes = False
            time.sleep(0.3)

        # ── 3. SAM Mask Override (action/contact relations) ──
        sam_override = None
        if rel.lower().strip() in SAM_CONTACT_RELATIONS:
            subj_det = find_best_detection(detections, subj)
            obj_det  = find_best_detection(detections, obj)
            if subj_det and obj_det:
                sm = masks.get(subj_det[0])
                om = masks.get(obj_det[0])
                if sm is not None and om is not None:
                    overlap = mask_overlap_ratio(sm, om)
                    if overlap > SAM_OVERLAP_THRESHOLD:
                        sam_override = "SUPPORTED"
                        print(f"    SAM mask overlap={overlap:.3f} -> override SUPPORTED")

        # ── Combine ──
        verdict, confidence = contrastive_verdict(main_yes, foil_yes)
        if sam_override == "SUPPORTED" and verdict != "HALLUCINATED":
            verdict = "SUPPORTED"
            confidence = "sam_confirmed"

        evidence = (f"OBSERVATION: {observation} | "
                    f"Main='{rel}': {'yes' if main_yes else 'no'} | "
                    f"Foil='{foil}': {'yes' if foil_yes else 'no'} | "
                    f"Confidence={confidence}")

        results.append((t, verdict, evidence))
        print(f"  ({subj},{rel},{obj}) -> {verdict} [{confidence}]")
        print(f"    Obs: {observation[:100]}")

    vlm_results[img_id] = results

print("\n=== Contrastive + Visual Evidence Verification Complete ===")


In [ ]:
# ============================================================
# Cell 8 — Stage 4: Merge Verdicts + Llama Correction
# ============================================================
# Llama text-only corrector receives Maverick's OBSERVATION text
# as structured evidence — much richer than before.
# No edit rate gate: corrections are evaluated qualitatively.

CORRECTION_PROMPT = """You are a precise caption editor making MINIMAL text corrections.

Original caption: "{caption}"

The following relations were verified as WRONG by a vision model.
For each, the vision model describes what it actually sees:

{hallucination_list}

Instructions:
1. Change ONLY the specific words describing each hallucinated relation
2. Keep ALL other words exactly the same
3. Use the vision model's observation to guide the correct relation word
4. If you must remove a claim, delete only that phrase — not the whole sentence
5. Do NOT add new information or restructure sentences
6. Output ONLY the corrected caption — no prefix, no explanation

Corrected caption:"""


def merge_verdicts(img_id):
    merged = []
    vlm_lookup = {(t["subject"], t["relation"], t["object"]): (v, e)
                  for t, v, e in vlm_results.get(img_id, [])}
    for t, verdict, evidence in spatial_results.get(img_id, []):
        key = (t["subject"], t["relation"], t["object"])
        if verdict in ("SUPPORTED", "HALLUCINATED"):
            merged.append((t, verdict, evidence))
        else:
            v, e = vlm_lookup.get(key, ("UNCERTAIN", "No verification"))
            merged.append((t, v, e))
    return merged


def correct_caption_llm(caption, hallucinated_triples):
    if not hallucinated_triples:
        return caption
    hall_list = ""
    for i, (t, evidence) in enumerate(hallucinated_triples, 1):
        # Extract observation from evidence string
        obs = ""
        if "OBSERVATION:" in evidence:
            obs_part = evidence.split("OBSERVATION:")[1]
            obs = obs_part.split("|")[0].strip()
        hall_list += (f"{i}. Triple: ({t['subject']}, {t['relation']}, {t['object']})\n"
                      f"   What vision model sees: {obs or 'relation not confirmed'}\n")

    prompt = CORRECTION_PROMPT.format(caption=caption, hallucination_list=hall_list)
    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0, max_tokens=400,
        )
        corrected = resp.choices[0].message.content.strip()
        corrected = re.sub(r'^(Corrected caption|Here is[^:]*)[:\s]+', '',
                           corrected, flags=re.IGNORECASE).strip()
        return corrected.strip('"').strip("'")
    except Exception as e:
        print(f"  Correction failed: {e}")
        return caption


def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = range(len(s2)+1)
    for c1 in s1:
        curr = [prev[0]+1]
        for j, c2 in enumerate(s2):
            curr.append(min(curr[-1]+1, prev[j+1]+1, prev[j]+(c1!=c2)))
        prev = curr
    return prev[-1]


# ── Run ──
final_results = {}
print("="*70)
print("RELCHECK V7 — RESULTS (Contrastive + Visual Evidence + SAM)")
print("="*70)

for img_id in images:
    caption = captions[img_id]
    merged = merge_verdicts(img_id)
    hallucinated = [(t, e) for t, v, e in merged if v == "HALLUCINATED"]
    supported    = [(t, e) for t, v, e in merged if v == "SUPPORTED"]
    uncertain    = [(t, e) for t, v, e in merged if v == "UNCERTAIN"]

    corrected = correct_caption_llm(caption, hallucinated) if hallucinated else caption
    edit_rate = levenshtein(caption, corrected) / max(len(caption), 1)

    final_results[img_id] = {
        "caption": caption, "corrected": corrected,
        "triples": len(merged), "supported": len(supported),
        "hallucinated": len(hallucinated), "uncertain": len(uncertain),
        "edit_rate": edit_rate, "details": merged,
    }

# Display
for img_id, r in final_results.items():
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.imshow(images[img_id]); ax.axis("off"); ax.set_title(img_id, fontsize=11, fontweight="bold")
    orig = textwrap.fill(f"Original:  {r['caption']}", width=90)
    corr = textwrap.fill(f"Corrected: {r['corrected']}", width=90)
    stats = (f"Triples: {r['triples']} | Supported: {r['supported']} | "
             f"Hallucinated: {r['hallucinated']} | Uncertain: {r['uncertain']} | "
             f"Edit: {r['edit_rate']:.1%}")
    hall_lines = [f"  X ({t['subject']}, {t['relation']}, {t['object']})"
                  for t, v, e in r["details"] if v == "HALLUCINATED"]
    txt = orig + "\n\n" + corr + "\n\n" + stats
    if hall_lines: txt += "\n\nHallucinated:\n" + "\n".join(hall_lines)
    fig.text(0.5, -0.02, txt, ha="center", va="top", fontsize=8, family="monospace",
             bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.8))
    plt.tight_layout(); plt.show(); print()


In [ ]:
# ============================================================
# Cell 9 — DECISION GATE: Viability Verdict
# ============================================================
print("\n" + "="*70)
print("VIABILITY ASSESSMENT — 15 IMAGE TEST (v7: SAM + Contrastive + Visual Evidence)")
print("="*70)

N = len(final_results)
n_hall  = sum(1 for r in final_results.values() if r["hallucinated"] > 0)
n_corr  = sum(1 for r in final_results.values() if r["corrected"] != r["caption"])
n_unch  = N - n_hall
tot_t   = sum(r["triples"]     for r in final_results.values())
tot_s   = sum(r["supported"]   for r in final_results.values())
tot_h   = sum(r["hallucinated"] for r in final_results.values())
tot_u   = sum(r["uncertain"]   for r in final_results.values())

corr_results = [r for r in final_results.values() if r["corrected"] != r["caption"]]
avg_edit = sum(r["edit_rate"] for r in corr_results) / max(len(corr_results), 1)

# Breakdown
geo_s  = sum(1 for r in final_results.values() for t,v,e in r["details"] if v=="SUPPORTED" and "geometry" in e.lower())
geo_h  = sum(1 for r in final_results.values() for t,v,e in r["details"] if v=="HALLUCINATED" and "geometry" in e.lower())
vlm_s  = sum(1 for r in final_results.values() for t,v,e in r["details"] if v=="SUPPORTED" and "OBSERVATION" in e)
vlm_h  = sum(1 for r in final_results.values() for t,v,e in r["details"] if v=="HALLUCINATED" and "OBSERVATION" in e)
sam_ov = sum(1 for r in final_results.values() for t,v,e in r["details"] if "sam_confirmed" in e)
hi_conf= sum(1 for r in final_results.values() for t,v,e in r["details"] if "high" in e)

print(f"\n  Images tested:           {N}")
print(f"  Images w/ hallucinations:{n_hall}/{N} ({n_hall/max(N,1):.0%})")
print(f"  Images corrected:        {n_corr}/{N}")
print(f"  Images unchanged:        {n_unch}/{N}")
print(f"  Total triples:           {tot_t}  ({tot_t/max(N,1):.1f}/image)")
print(f"  Supported:               {tot_s} ({tot_s/max(tot_t,1):.0%})")
print(f"  Hallucinated:            {tot_h} ({tot_h/max(tot_t,1):.0%})")
print(f"  Uncertain:               {tot_u} ({tot_u/max(tot_t,1):.0%})")
print(f"\n--- Verification Breakdown ---")
print(f"  Geometry SUPPORTED:      {geo_s}")
print(f"  Geometry HALLUCINATED:   {geo_h}")
print(f"  VLM SUPPORTED:           {vlm_s}")
print(f"  VLM HALLUCINATED:        {vlm_h}")
print(f"  SAM mask confirmations:  {sam_ov}")
print(f"  High-confidence verdicts:{hi_conf}/{tot_t} ({hi_conf/max(tot_t,1):.0%})")
print(f"\n--- Correction Quality ---")
print(f"  Avg edit rate (corrected): {avg_edit:.1%}")

# ── Qualitative rating ──
print("\n" + "="*70)
print("RATE EACH CORRECTED IMAGE: good (g) / bad (b) / skip (s)")
print("="*70)
good, bad = 0, 0
for img_id, r in final_results.items():
    if r["corrected"] != r["caption"]:
        rating = input(
            f"[{img_id[:8]}]\n"
            f"  Before: {r['caption'][:80]}...\n"
            f"  After:  {r['corrected'][:80]}...\n"
            f"  Edit: {r['edit_rate']:.0%} | Hallucinated: {r['hallucinated']}/{r['triples']}\n"
            f"  Rating (g/b/s): "
        ).strip().lower()
        if rating == "g": good += 1
        elif rating == "b": bad += 1

total_rated = good + bad
pct_good = good / max(total_rated, 1)
print(f"\n  Good corrections: {good}/{total_rated} ({pct_good:.0%})")
print(f"  Bad corrections:  {bad}")

print("\n" + "="*70)
if pct_good >= 0.75 and n_hall >= 5:
    print("  VERDICT: VIABLE — proceed to full 600-image pipeline")
elif pct_good >= 0.60:
    print("  VERDICT: MARGINAL — review bad cases, then proceed")
else:
    print("  VERDICT: NOT VIABLE — check: false positive rate? VLM answers?")
print("="*70)

import json as _j
with open("viability_v7_results.json", "w") as f:
    _j.dump({k: {**v, "details": [(t,vv,e) for t,vv,e in v["details"]]}
             for k, v in final_results.items()}, f, indent=2, default=str)
print("\nSaved to viability_v7_results.json")
